In [1]:
import pandapower.plotting as pplt
import matplotlib.pyplot as plt
from pandapower_env.data.example_configs import config_case89, config_case30
from pandapower.networks import case14, case30
from pandapower_env.toolbox.plotting_helpers import (
    calculate_externals_locations,
    create_bus2bus_geodata,
    create_geo_column_from_xy,
    create_xy_columns_from_geo,
    label_buses,
    label_lines,
    label_substations,
    plot_all_peripherals,
)
from pandapower_env.environments.simulation_env import PPTopoGym
from functools import partial
import pandas as pd
import copy
import numpy as np
from pandapower_env.agents.benchmark_agents import DoNothingAgent, GreedyAgent
from gymnasium import spaces
import time

In [2]:
config30 = config_case30() # line 9 is overloaded plotting in action_explore branch, needs update
env = PPTopoGym(config30)
net = config30["net"]
net.trafo

,name,std_type,hv_bus,lv_bus,sn_mva,vn_hv_kv,vn_lv_kv,vk_percent,vkr_percent,pfe_kw,...,tap_min,tap_max,tap_step_percent,tap_step_degree,tap_pos,tap_phase_shifter,parallel,df,in_service,tap_changer_type


In [3]:
cfg_dn = copy.deepcopy(config30)
obs, info = env.reset(options={"index": 0})
action_space = spaces.Discrete(len(cfg_dn["action_space"]))
agent = DoNothingAgent(action_space)
total_episodes_overloaded = []
print("Started using greedy agent on episodes")
for episode in range(0,10):
    env.reset(options={"index":episode})
    max_loading_episode = 0
    steps_overloaded = 0 
    done = False
    while not done:
        start = time.time()
        action = agent.act(obs)
        end = time.time()
        #print(f"greedy step took: {end - start:.4f}sec")
        obs, reward, terminated, truncated, info = env.step(action)
        overload = obs["line_loadings"].max()
        #print(overload, obs["line_loadings"])
        max_loading_episode = max(overload, max_loading_episode)
        if overload >= 100.0:
            steps_overloaded += 1
        done = terminated or truncated
    number_steps = len(env.log_actions)
    print(f"A total of {number_steps} steps reached.")
    print(f"Episode {episode} had maximal overload of {max_loading_episode}, and total {steps_overloaded} steps overloaded")
    if max_loading_episode > 100:
        total_episodes_overloaded.append(episode)

Started using greedy agent on episodes
A total of 96 steps reached.
Episode 0 had maximal overload of 78.90223693847656, and total 0 steps overloaded
A total of 96 steps reached.
Episode 1 had maximal overload of 78.90223693847656, and total 0 steps overloaded
A total of 96 steps reached.
Episode 2 had maximal overload of 78.90223693847656, and total 0 steps overloaded
A total of 96 steps reached.
Episode 3 had maximal overload of 78.90223693847656, and total 0 steps overloaded
A total of 96 steps reached.
Episode 4 had maximal overload of 78.90223693847656, and total 0 steps overloaded
A total of 96 steps reached.
Episode 5 had maximal overload of 78.90223693847656, and total 0 steps overloaded
A total of 96 steps reached.
Episode 6 had maximal overload of 78.90223693847656, and total 0 steps overloaded
A total of 96 steps reached.
Episode 7 had maximal overload of 78.90223693847656, and total 0 steps overloaded
A total of 96 steps reached.
Episode 8 had maximal overload of 78.9022369

In [ ]:
cfg_greedy = copy.deepcopy(config30)
obs, info = env.reset(options={"index": 0})
done = False
action_space = spaces.Discrete(len(cfg_greedy["action_space"]))
agent = GreedyAgent(action_space, cfg_greedy,n_workers = 100)
total_episodes_overloaded = []
print("Started using greedy agent on episodes")
for episode in range(0,10):
    max_loading_episode = 0
    steps_overloaded = 0 
    done = False
    obs, info = env.reset(options={"index":episode})
    while not done:
        start = time.time()
        action = agent.act(obs, info)
        end = time.time()
        print(f"greedy took action {action} at step {len(env.log_actions)}, time: {end - start:.4f}sec")
        obs, reward, terminated, truncated, info = env.step(action)
        overload = obs["line_loadings"].max()
        max_loading_episode = max(overload, max_loading_episode)
        if overload >= 100.0:
            steps_overloaded += 1
        done = terminated or truncated
    number_steps = len(env.log_actions)
    print(f"Episode {episode} had maximal overload of {max_loading_episode}, and total {steps_overloaded} steps overloaded")
    print(f"A total of {number_steps} steps reached.")
    if max_loading_episode > 100:
        total_episodes_overloaded.append(episode)

Started using greedy agent on episodes
greedy took action 75 at step 0 took: 10.2565sec
greedy took action 21 at step 1 took: 0.6956sec
greedy took action 46 at step 2 took: 0.4431sec
greedy took action 94 at step 3 took: 0.3692sec
greedy took action 77 at step 4 took: 0.3568sec
greedy took action 178 at step 5 took: 0.3374sec
greedy took action 218 at step 6 took: 0.3316sec
greedy took action 283 at step 7 took: 0.3424sec
greedy took action 253 at step 8 took: 0.3641sec
greedy took action 284 at step 9 took: 0.3346sec
greedy took action 212 at step 10 took: 0.3507sec
greedy took action 218 at step 11 took: 0.3678sec
greedy took action 343 at step 12 took: 0.3192sec
greedy took action 173 at step 13 took: 0.3478sec
greedy took action 184 at step 14 took: 0.3423sec
greedy took action 344 at step 15 took: 0.3394sec
greedy took action 182 at step 16 took: 0.3267sec
greedy took action 181 at step 17 took: 0.3423sec
greedy took action 166 at step 18 took: 0.3374sec
greedy took action 148 at